# 03 - ETL Spark: `.jpg` -> `.parquet` (Pipeline B)

**Por quê:** o notebook 02 diagnosticou que o gargalo é a **CPU decodificando JPEG em tempo real** (decode + resize a cada época), deixando a GPU faminta (~27% ociosa).

**Solução (ETL):** usar o **Apache Spark** pra decodificar + redimensionar (224×224×3) as imagens **uma única vez** e gravar os pixels prontos em **`.parquet` particionado**. Na inferência (notebook 04) a CPU só **lê o array** (sem decode) e o normaliza barato antes de enviar pra GPU.

**Formato: `uint8`** (pixels crus 0-255), serializado como bytes numa coluna `BinaryType`. É **sem perda** (o pixel que sai do decode já é uint8) e **metade** do float16 (~150 KB/imagem → ~7-8 GB pras 70k). O `preprocess` do MobileNet (`x/127.5 - 1`) fica pro read do nb 04 — custo trivial (um `cast+scale` vetorizado, nada perto do decode que eliminamos). O ETL roda **100% na CPU**.

> Roda dentro do contêiner. Saída em `/tf/data/optimized/`. Spark UI em http://localhost:4040.

In [1]:
import time
from pathlib import Path

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import BinaryType

RAW_DIR       = "/tf/data/raw_jpg"     # imagens .jpg de entrada
OUT_DIR       = "/tf/data/optimized"   # parquet de saida (Pipeline B)
IMG_SIZE      = 224                    # dimensao de entrada da MobileNetV2
LIMIT         = None                   # None = TODAS as ~70k (~7-8GB uint8). Um inteiro p/ teste rapido.
IMGS_PER_FILE = 2000                   # alvo de imagens por arquivo parquet

spark = (
    SparkSession.builder
    .appName("etl_jpg_to_parquet_uint8")
    .master("local[2]")                                              # 2 vCPU do conteiner
    .config("spark.driver.memory", "4g")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "512")   # uint8 e' menor -> batch pode ser maior
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version, "| driver.memory:", spark.conf.get("spark.driver.memory"))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/01 05:02:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.0 | driver.memory: 4g


In [2]:
# Le as imagens como binario: colunas path, modificationTime, length, content (bytes).
df = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.jpg")
    .load(RAW_DIR)
)

# label pelo prefixo do nome do arquivo (cat.* -> 0, dog.* -> 1)
df = df.withColumn("fname", F.element_at(F.split(F.col("path"), "/"), -1))
df = df.withColumn("label", F.when(F.lower(F.col("fname")).startswith("cat"), 0).otherwise(1))

if LIMIT is not None:
    df = df.limit(LIMIT)

n_total = df.count()
n_part  = max(2, round(n_total / IMGS_PER_FILE))
print(f"Imagens a processar: {n_total:,}  ->  {n_part} arquivos parquet")

# repartition ANTES do decode: espalha o trabalho pesado nas tasks E ja define o nº de arquivos
# de saida. Embaralha so os bytes JPEG (~1.7GB), nao os pixels decodificados.
df = df.repartition(n_part)

Imagens a processar: 70,433  ->  35 arquivos parquet


In [3]:
# UDF: JPEG bytes -> PIL decode -> resize 224 -> pixels uint8 (0-255) -> bytes.
# Usa Pillow (leve, seguro no worker Spark). uint8 e' SEM PERDA (o pixel decodificado ja e uint8)
# e metade do float16. O preprocess do MobileNet (x/127.5 - 1) fica pro read do nb 04.
# Imagem corrompida -> array de zeros (fallback, nao derruba a task).
@pandas_udf(BinaryType())
def decode_to_uint8(content: pd.Series) -> pd.Series:
    import io
    import numpy as np
    from PIL import Image

    out = []
    for buf in content:
        try:
            img = Image.open(io.BytesIO(buf)).convert("RGB").resize((224, 224), Image.BILINEAR)
            arr = np.asarray(img, dtype=np.uint8)          # (224,224,3) em 0-255, sem perda
            out.append(arr.tobytes())
        except Exception:
            out.append(np.zeros((224, 224, 3), np.uint8).tobytes())
    return pd.Series(out)

# colunas finais: label + data (pixels uint8 224x224x3 serializados em bytes)
df_out = df.select("label", decode_to_uint8(F.col("content")).alias("data"))

In [4]:
# Escreve o parquet particionado. SEM repartition aqui (ja foi antes do decode) -> n_part arquivos.
t0 = time.time()
df_out.write.mode("overwrite").parquet(OUT_DIR)
etl_secs = time.time() - t0
print(f"ETL concluido em {etl_secs:.0f}s  ->  {OUT_DIR}")

ETL concluido em 56s  ->  /tf/data/optimized


In [5]:
# Validacao: le de volta, confere contagem, reshape/dtype/range de 1 registro e tamanho em disco.
import numpy as np
import subprocess

back = spark.read.parquet(OUT_DIR)
n_out = back.count()
print(f"Registros no parquet : {n_out:,}")
print(f"Schema               : {back.schema.simpleString()}")

row = back.limit(1).collect()[0]
arr = np.frombuffer(row["data"], dtype=np.uint8).reshape(IMG_SIZE, IMG_SIZE, 3)
print(f"1 registro           : label={row['label']} | shape={arr.shape} | dtype={arr.dtype} "
      f"| range=[{arr.min()}, {arr.max()}]  (esperado 0-255)")

sz = subprocess.run(["du", "-sh", OUT_DIR], capture_output=True, text=True).stdout.split()[0]
n_files = len(list(Path(OUT_DIR).glob("*.parquet")))
print(f"Tamanho em disco     : {sz}  em {n_files} arquivos parquet")
print(f"Bytes por imagem     : {IMG_SIZE*IMG_SIZE*3:,} (uint8)")

Registros no parquet : 70,433
Schema               : struct<label:int,data:binary>
1 registro           : label=1 | shape=(224, 224, 3) | dtype=uint8 | range=[1, 255]  (esperado 0-255)
Tamanho em disco     : 9.4G  em 35 arquivos parquet
Bytes por imagem     : 150,528 (uint8)


In [6]:
spark.stop()
print("Spark encerrado. Parquet uint8 pronto p/ o notebook 04 (benchmark do Pipeline B).")

Spark encerrado. Parquet uint8 pronto p/ o notebook 04 (benchmark do Pipeline B).
